# Llama 3.1 8B SFT v3 — LCAE AI MC (Kaggle)

현재 LLM 플로우에 맞춘 `train_v3.jsonl` 전용 학습 노트북입니다.

**실행 전 체크리스트**
1. 오른쪽 패널 → Accelerator: GPU T4/P100 선택
2. Add-ons → Secrets → `HF_TOKEN` 등록 후 활성화
3. Add-ons → Datasets → `train_v3.jsonl`이 들어있는 데이터셋 추가
4. 셀 순서대로 실행

학습 산출물은 `/kaggle/working/adapters_v3`에 저장됩니다.


In [ ]:
# 1. GPU + 디스크 확인
import shutil, torch
assert torch.cuda.is_available(), 'GPU가 없습니다. Accelerator를 GPU로 변경하세요.'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Disk : {free / 1e9:.1f} GB 여유')


In [ ]:
# 2. 패키지 설치
!pip install -q \
    "transformers>=4.45.0" \
    "trl==1.5.1" \
    "peft>=0.13.0" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.20.0"
print('설치 완료')


In [ ]:
# 3. HuggingFace 인증
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('HF_TOKEN 로드 성공')
except Exception:
    print('Secrets에 HF_TOKEN 없음 → 수동 입력')
    login()


In [ ]:
# 4. train_v3.jsonl 자동 탐색
import glob, os

candidates = glob.glob('/kaggle/input/**/train_v3.jsonl', recursive=True)
assert candidates, 'train_v3.jsonl을 찾을 수 없습니다. Add-ons → Datasets에 v3 데이터를 추가하세요.'
DATA_PATH = candidates[0]
print(f'데이터 경로: {DATA_PATH}')
!wc -l {DATA_PATH}


In [ ]:
# 5. 데이터 스키마 검증
import json
from collections import Counter

rows = []
errors = []
with open(DATA_PATH, encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        obj = json.loads(line)
        msgs = obj.get('messages', [])
        if [m.get('role') for m in msgs] != ['system', 'user', 'assistant']:
            errors.append((i, 'roles'))
            continue
        out = json.loads(msgs[2]['content'])
        if set(out.keys()) != {'topic', 'intent', 'mc_script'}:
            errors.append((i, 'keys', out))
        if 'summary' in msgs[2]['content'] or '[이전 요약]' in msgs[1]['content'] or '[대기 질문]' in msgs[1]['content']:
            errors.append((i, 'legacy'))
        rows.append(out)

print(f'rows={len(rows)} errors={len(errors)}')
print('intent:', Counter(r['intent'] for r in rows))
print('nonempty mc_script:', Counter(r['intent'] for r in rows if r['mc_script']))
assert not errors, errors[:10]


In [ ]:
# 6. QLoRA 학습
import json, os, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL     = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
OUTPUT_DIR     = '/kaggle/working/adapters_v3'
LORA_RANK      = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3
MAX_SEQ_LENGTH = 1024
SAVE_STEPS     = 100
LOGGING_STEPS  = 10

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return Dataset.from_list(records)

def format_chat(example, tokenizer):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

print('[1/4] 토크나이저 로드...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.model_max_length = MAX_SEQ_LENGTH

print('[2/4] 데이터 로드...')
raw_dataset = load_jsonl(DATA_PATH)
dataset = raw_dataset.map(lambda ex: format_chat(ex, tokenizer))
print(f'  총 {len(dataset)}개 샘플')
print(dataset[0]['text'][:500])

print('[3/4] 모델 로드 (4-bit QLoRA)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print('[4/4] 학습 시작...')
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=25,
    bf16=False,
    fp16=True,
    dataset_text_field='text',
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to='none',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=0,
)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)
for name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype in (torch.bfloat16, torch.float16):
        param.data = param.data.to(torch.float32)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'완료: {OUTPUT_DIR}')


In [ ]:
# 7. 어댑터 압축 (Kaggle Output에서 다운로드하거나 Dataset으로 만들기)
import os, shutil
ARCHIVE = '/kaggle/working/adapters_v3'
zip_path = shutil.make_archive(ARCHIVE, 'zip', OUTPUT_DIR)
print(f'압축 완료: {zip_path}')
!ls -lh /kaggle/working/adapters_v3.zip
print('Kaggle Output 탭에서 adapters_v3.zip을 다운로드하거나 새 Dataset으로 저장하세요.')
